In [ ]:
import pandas as pd

# Load the dataset
datadf = pd.read_csv(
    '/content/u.data',
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
datadf = datadf [['user_id', 'item_id', 'rating']]
print(datadf.head())

   user_id  item_id  rating
0      196      242       3
1      186      302       3
2       22      377       1
3      244       51       2
4      166      346       1


In [ ]:
def ndcg_at_k(test, predicted_matrix, train_matrix, k=10, mean_rating=0):
    ndcg_scores = []

    # group test ratings by user
    test_grouped = test.groupby('user_id')

    for user_id, user_test in test_grouped:
        if user_id not in train_matrix.index:
            continue

        user_idx = train_matrix.index.get_loc(user_id)

        # get predicted scores for all items in test set for this user
        actual_ratings = []
        predicted_scores = []

        for _, row in user_test.iterrows():
            item_id = row['item_id']
            actual = row['rating']

            if item_id in train_matrix.columns:
                item_idx = train_matrix.columns.get_loc(item_id)
                predicted = predicted_matrix[user_idx, item_idx]
            else:
                predicted = mean_rating

            actual_ratings.append(actual)
            predicted_scores.append(predicted)

        if len(actual_ratings) < 2:
            continue

        # sort by predicted score to get top K
        paired = sorted(zip(predicted_scores, actual_ratings),
                       reverse=True)[:k]

        # compute DCG
        dcg = sum(rating / np.log2(rank + 2)
                 for rank, (_, rating) in enumerate(paired))

        # compute ideal DCG
        ideal_paired = sorted(actual_ratings, reverse=True)[:k]
        idcg = sum(rating / np.log2(rank + 2)
                  for rank, rating in enumerate(ideal_paired))

        if idcg > 0:
            ndcg_scores.append(dcg / idcg)

    return np.mean(ndcg_scores) if ndcg_scores else 0.0

In [ ]:
def laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 4.0
    noise = np.random.laplace(0, sensitivity/epsilon, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
import numpy as np

def piecewise_mechanism(x, epsilon):
    # Step 1: normalise x from [1,5] to [-1,1]
    x_norm = 2 * (x - 1) / (5 - 1) - 1
    # C = (e^(ε/2) + 1) / (e^(ε/2) - 1)
    c= (np.exp(epsilon/2)+1)/(np.exp(epsilon/2)-1)
    # p = (e^ε - e^(ε/2)) / (2 * e^(ε/2) + 2)
    p= (np.exp(epsilon)-np.exp(epsilon/2))/(2*np.exp(epsilon/2)+2)
    # l(x) = (C+1)/2 * x_norm - (C-1)/2
    # r(x) = l(x) + C - 1
    l= (c+1)/2*x_norm-(c-1)/2
    r=l+c-1
    # threshold = e^(ε/2) / (e^(ε/2) + 1)
    threshold = np.exp(epsilon/2)/(np.exp(epsilon/2)+1)
    if np.random.uniform(0, 1) < threshold:
    # heads — sample from centre zone
      output = np.random.uniform(l, r)
    else:
    # tails — sample from tails
    # left tail runs from -C to l
    # right tail runs from r to +C
    # pick one randomly
      output = np.random.uniform(-c, l) if np.random.uniform(0,1) < 0.5 else np.random.uniform(r, c)
    x_out = (output + 1) * 2 + 1
    x_out = np.clip(x_out, 1, 5)
    return x_out

In [ ]:
def apply_piecewise(matrix, epsilon, mean_rating):
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                noisy_matrix[i, j] = piecewise_mechanism(matrix[i, j], epsilon)
    return noisy_matrix

In [ ]:
def gaussian_mechanism(matrix, epsilon, delta, mean_rating):
    sensitivity = 4.0
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = np.random.normal(0, sigma, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
def bounded_laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 1.0  # normalised scale [0,1]
    b = sensitivity / epsilon
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                r = (matrix[i, j] - 1) / 4  # normalise to [0,1]
                while True:
                    noise = np.random.laplace(0, b)
                    r_noisy = r + noise
                    if 0 <= r_noisy <= 1:
                        break
                noisy_matrix[i, j] = r_noisy * 4 + 1  # denormalise
    return noisy_matrix

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

n_splits = 10
datadf_50 = datadf.sample(frac=0.5, random_state=42)
# datadf_20 = datadf.sample(frac=0.2, random_state=42)
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(kf.split(datadf_50)):
    train = datadf_50.iloc[train_idx]
    test = datadf_50.iloc[test_idx]
all_results = []
epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sparsity_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(datadf_50)):
    print(f"Fold {fold+1}/{n_splits}")

    # split data
    train = datadf_50.iloc[train_idx]
    test = datadf_50.iloc[test_idx]
    print(f"Train size: {len(train)}, Test size: {len(test)}")

    try:
        train_matrix = train.pivot(index='user_id',
                                   columns='item_id',
                                   values='rating')
        print(f"Train matrix shape: {train_matrix.shape}")
    except Exception as e:
        print(f"Error building pivot at fold {fold}: {e}")
        continue
    # build matrix
    #train_matrix = train.pivot(index='user_id',
    #                           columns='item_id',
    #                           values='rating')
    train_array = train_matrix.values.astype(float)
    mean_rating = np.nanmean(train_array)
    print(f"Mean rating: {mean_rating:.4f}, NaN count: {np.isnan(train_array).sum()}")
    train_array[np.isnan(train_array)] = mean_rating
    print(f"NaN after fill: {np.isnan(train_array).sum()}")

    test_users = test['user_id'].values
    test_items = test['item_id'].values
    test_ratings = test['rating'].values
    gaussian_with_delta = lambda matrix, eps, mean_rating: gaussian_mechanism(matrix, eps, 1e-5, mean_rating)
    for eps in epsilons:
      for mechanism_name, mechanism_fn in [
        ('clipped_laplace', laplace_mechanism),
        ('bounded_laplace', bounded_laplace_mechanism),
        ('gaussian', gaussian_with_delta),
        ('piecewise', apply_piecewise)
      ]:
        try:
            noisy = mechanism_fn(train_array, eps, mean_rating)
            noisy = np.nan_to_num(noisy, nan=mean_rating,
                                  posinf=5.0, neginf=1.0)

            svd = TruncatedSVD(n_components=20, random_state=42)
            reduced = svd.fit_transform(noisy)
            predicted_matrix = np.dot(reduced, svd.components_)
            predicted_matrix = np.nan_to_num(predicted_matrix,
                                             nan=mean_rating)

            predicted_ratings = []
            for user, item in zip(test_users, test_items):
                if user in train_matrix.index and item in train_matrix.columns:
                    u_idx = train_matrix.index.get_loc(user)
                    i_idx = train_matrix.columns.get_loc(item)
                    predicted_ratings.append(predicted_matrix[u_idx, i_idx])
                else:
                    predicted_ratings.append(mean_rating)

            predicted_ratings = np.array(predicted_ratings)
            predicted_ratings = np.nan_to_num(predicted_ratings,
                                              nan=mean_rating)
            rmse = np.sqrt(mean_squared_error(test_ratings,
                                              predicted_ratings))
            ndcg = ndcg_at_k(test, predicted_matrix, train_matrix, k=10,
                  mean_rating=mean_rating)

            all_results.append({
              'fold': fold,
              'epsilon': eps,
              'mechanism': mechanism_name,
              'rmse': rmse,
              'ndcg': ndcg
            })

            print(f"Fold {fold+1}, {mechanism_name}, ε={eps}: RMSE={rmse:.4f}, NDCG@10={ndcg:.4f}")

        except Exception as e:
            print(f"ERROR at fold {fold+1}, {mechanism_name}, ε={eps}: {e}")
            continue

# compute mean and std across folds
results_df = pd.DataFrame(all_results)
summary = results_df.groupby(['epsilon', 'mechanism']).agg(
    rmse_mean=('rmse', 'mean'),
    rmse_std=('rmse', 'std'),
    ndcg_mean=('ndcg', 'mean'),
    ndcg_std=('ndcg', 'std')
).reset_index()
print(summary)

Fold 1/10
Train size: 45000, Test size: 5000
Train matrix shape: (943, 1573)
Mean rating: 3.5250, NaN count: 1438339
NaN after fill: 0
Fold 1, clipped_laplace, ε=0.1: RMSE=1.3230, NDCG@10=0.9144
Fold 1, bounded_laplace, ε=0.1: RMSE=1.1448, NDCG@10=0.9162
Fold 1, gaussian, ε=0.1: RMSE=1.3432, NDCG@10=0.9161
Fold 1, piecewise, ε=0.1: RMSE=1.1446, NDCG@10=0.9225
Fold 1, clipped_laplace, ε=0.5: RMSE=1.2766, NDCG@10=0.9194
Fold 1, bounded_laplace, ε=0.5: RMSE=1.1458, NDCG@10=0.9163
Fold 1, gaussian, ε=0.5: RMSE=1.3196, NDCG@10=0.9194
Fold 1, piecewise, ε=0.5: RMSE=1.1363, NDCG@10=0.9236
Fold 1, clipped_laplace, ε=1.0: RMSE=1.2363, NDCG@10=0.9171
Fold 1, bounded_laplace, ε=1.0: RMSE=1.1367, NDCG@10=0.9179
Fold 1, gaussian, ε=1.0: RMSE=1.3153, NDCG@10=0.9200
Fold 1, piecewise, ε=1.0: RMSE=1.1294, NDCG@10=0.9235
Fold 1, clipped_laplace, ε=2.0: RMSE=1.2010, NDCG@10=0.9138
Fold 1, bounded_laplace, ε=2.0: RMSE=1.1332, NDCG@10=0.9160
Fold 1, gaussian, ε=2.0: RMSE=1.2943, NDCG@10=0.9184
Fold 1, pie

In [ ]:
results_df.to_excel('ldp_results_natural.xlsx', index=False)
summary.to_excel('ldp_summary_natural.xlsx', index=False)

In [ ]:
results_df.to_excel('ldp_results_50pct.xlsx', index=False)
summary.to_excel('ldp_summary_50pct.xlsx', index=False)

In [ ]:
results_df.to_excel('ldp_results_20pct.xlsx', index=False)
summary.to_excel('ldp_summary_20pct.xlsx', index=False)